# Transit available versus walking only: eleven cities

Paired comparisons for White alone, Black alone, Hispanic/Latino any race, and Asian alone. These categories overlap. Both scenarios use the same 4,738 populated 2020 city-limit tract portions and the same demographic counts and representative points. All cells read saved local data; no downloads or Routes API calls.

**Walking is held fixed:** complete great-circle distances × 1.3 at 4.8 km/h. The transit scenario adds the saved rail/bus expected-time candidates; walking-only removes every transit candidate. This differs from the previous report's transit-dependent walking fallback rule, so both sides are recomputed. The old scores are preserved separately. Distance estimates may cross water and are not verified street itineraries.

Shared economic parameters: $15/hour, beta=0.1 per dollar, local floor=$1.25; transit carries an illustrative standardized $3 fare. Transit schedules are October 20, 2026 except San Francisco August 25, 2026. This is a fixed-population model comparison, not a causal forecast.


## Definitions

$$C_{ij}=\max\{c_0,\min_m[p^m_{ij}+\omega t^m_{ij}]\},\quad F_{ij}=e^{-\beta C_{ij}},\quad P_{ij}=\frac{F_{ij}D_j}{\sum_k F_{ik}D_k},\quad A_r=P\,\mathrm{diag}(p_r).$$

Time is in hours and cost in dollars. $K$ is the category's city population fraction, unchanged across scenarios.

| Displayed metric | Calculation |
|---|---|
| PSSI | Dominant eigenvalue $\lambda$ of $A_r$ |
| Normalized PSSI | $(\lambda-K)/(1-K)$ |
| Fryer SSI | Existing majority-tract adaptation on $P$: induced matrices for $p_i\geq0.5$, component eigenvalues averaged by majority-tract counts |
| Adjusted Fryer | $SSI-K$, with no division |

The adjusted Fryer column uses the requested population-share subtraction, not the paper's simulated null baseline. No-majority-tract cases have raw SSI=0 by the existing convention and adjusted value $-K$; asterisks flag them in the plots. Nonmajority tracts have undefined local Fryer scores. Negative adjustments are retained.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/transit_comparison.py').exists())
BASE = ROOT / 'outputs/multicity/transit_comparison'
scores = pd.read_csv(BASE / 'scenario_scores.csv')
paired = pd.read_csv(BASE / 'paired_comparison.csv')
manifest = json.loads((BASE / 'manifest.json').read_text())
verification = json.loads((BASE / 'verification.json').read_text())
print(f'{scores.city.nunique()} cities; {len(scores)} scenario/category results; {len(paired)} paired comparisons')
print('Matrix verification:', verification['passed'])
print('Report:', BASE / 'index.html')


## Four paired plots

For each metric, both scenarios share the same color scale. The categories and city order are identical. A figure's cells show the actual scores, rounded to three decimal places; tables retain full precision.


In [ ]:
metric_names = ['segregation_index', 'segregation_index_normalized', 'fryer_majority_ssi_minus_K', 'fryer_majority_ssi']
for metric in metric_names:
    fig, ax = plt.subplots(figsize=(15, 9))
    ax.imshow(plt.imread(BASE / f'{metric}.png'))
    ax.axis('off')
    plt.tight_layout()
    plt.show()


## Values, differences and column definitions

Every difference is **walking only minus walking + transit**. Positive differences mean a higher score after removing transit under these assumptions. Generalized costs can only increase when transit is removed, but row normalization means the direction of a segregation score change is not guaranteed by that fact alone.


In [ ]:
with pd.option_context('display.max_columns', None, 'display.max_rows', None):
    display(paired)
    display(pd.read_csv(BASE / 'paired_column_dictionary.csv'))

for metric in metric_names:
    print(metric, '— change without minus with')
    display(paired.pivot(index='city', columns='group', values=metric + '_change_without_minus_with'))


## Tract eigenvectors and local Fryer scores

There is one PSSI eigenvalue per city/category/scenario. Each tract has a component of its sum-one right eigenvector. The exported $n v_i$ values have mean one. Fryer component eigenvectors also sum to one; local SSI is component size × component eigenvalue × eigenvector entry. Its mean across majority tracts equals city SSI.

Change `city_id` and `group` below to inspect another result. Small isolated tract portions can strongly influence spectral scores, so the population and partial-tract columns are retained.


In [ ]:
city_id = 'chicago'
group = 'black_alone'
tracts = pd.read_csv(BASE / 'all_tract_scores.csv', dtype={'tract_geoid': str})
selected = tracts[tracts.city_id.eq(city_id) & tracts.group.eq(group)]
tract_fields = ['matrix_index', 'tract_geoid', 'population', 'group_proportion', 'pssi_city_eigenvalue',
                'pssi_right_eigenvector', 'pssi_eigenvector_times_n', 'fryer_city_ssi',
                'fryer_is_majority_tract', 'fryer_component_eigenvalue', 'fryer_tract_ssi']
for scenario in ['with_transit', 'walking_only']:
    print(scenario)
    display(selected[selected.scenario.eq(scenario)][tract_fields].sort_values('pssi_right_eigenvector', ascending=False).head(15))


## Cost matrices and verification

All city/scenario folders contain cost matrices (C, F, walking/transit costs, selected mode, water flags) and analysis matrices (P, four A matrices, eigenpairs, counts, GEOID ordering). The independent verification reconstructs P and A from the defining formulas and checks all eigenpair residuals, normalization identities and local Fryer aggregation.


In [ ]:
with np.load(BASE / city_id / 'with_transit/cost_matrices.npz') as on, np.load(BASE / city_id / 'walking_only/cost_matrices.npz') as off:
    np.testing.assert_array_equal(on['walk_cost_usd'], off['walk_cost_usd'])
    assert (on['C_usd'] <= off['C_usd'] + 1e-12).all()
    assert not (off['chosen_mode'] == 'TRANSIT').any()
    print('Shared walking estimates and transit toggle verified for', city_id)
    display(pd.DataFrame(on['C_usd'][:5, :5]))
    display(pd.DataFrame(off['C_usd'][:5, :5]))
display(verification)


## Approximation diagnostics and previous report

Water flags come from straight chords intersecting mapped water. They can identify shortcuts but also flag legitimate trips via bridges or indirect routes. Flagged probability mass is not a bound on eigenvalue error. The previous report's different walking-completion policy is quantified in the reference table below; its with-transit scores should not be directly paired with this walking-only scenario.


In [ ]:
display(pd.read_csv(BASE / 'walking_assumptions.csv'))
display(pd.read_csv(BASE / 'previous_report_reference.csv'))
print('Parameters:', manifest['parameters'])
print('Transit source coverage:', ROOT / 'outputs/multicity/transit_sources.json')
print('Required dataset attribution:', ROOT / 'outputs/multicity/data_attribution.txt')


## Regenerate locally

Run from the project directory. The first command reuses saved Census/GTFS duration matrices; it does not route again or request network data.

```bash
MPLCONFIGDIR=/private/tmp/apma4903-matplotlib .venv/bin/python -m scripts.transit_comparison
.venv/bin/python -m scripts.verify_transit_comparison
.venv/bin/python -m unittest discover -s tests -q
```

The solver uses verified power iteration and a dense eigenvalue fallback for nearly disconnected matrices that converge too slowly. All saved eigenpairs must pass the same residual tolerance.
